In [0]:
%sql
-- CREATE SCHEMA
create database if not exists telco_schema;

-- =========================================
-- 1. CREATE TABLE
-- =========================================
DROP TABLE IF EXISTS telco_schema.dim_device;
CREATE TABLE telco_schema.dim_device (
    device_id INT PRIMARY KEY,
    device_type VARCHAR(50),
    brand VARCHAR(50),
    model VARCHAR(50),
    os VARCHAR(50),
    owner_customer_id INT,
    status VARCHAR(20),
    updated_at TIMESTAMP  
);

-- =========================================
-- 2. FIRST RUN (INITIAL LOAD)
-- =========================================
INSERT INTO telco_schema.dim_device (device_id, device_type, brand, model, os, owner_customer_id, status, updated_at) VALUES
(1001, 'Smartphone', 'Apple', 'iPhone 15', 'iOS', 6566, 'Active', NOW()),
(1002, 'Smartphone', 'Samsung', 'Galaxy S23', 'Android', 6807, 'Active', NOW()),
(1003, 'Tablet', 'Google', 'Pixel Tablet', 'Android', 8573, 'Active', NOW()),
(1004, 'FeaturePhone', 'Nokia', 'Nokia 3310', 'S30+', 5724, 'Active', NOW()),
(1005, 'Smartphone', 'Google', 'Pixel 8', 'Android', 9550, 'Active', NOW());

In [0]:
container = "source"
storage_account = "sourcesystemadlsgen2"
source_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

In [0]:
df1 = spark.read.table("telecom_7405608425992653.telco_schema.dim_device").where("updated_at >coalesce((select max(updated_at) from telecom_7405608425992653.telco_bronze.device_bronze_table),'1970-01-01')")
# df1.show()
df1.write.format("parquet").save(f"{source_path}/device_source/device1")

In [0]:
%sql
-- =========================================
-- 3. SECOND RUN (CDC CHANGES)
-- =========================================

-- Inserts: New devices detected in the stream
INSERT INTO telco_schema.dim_device (device_id, device_type, brand, model, os, owner_customer_id, status, updated_at) VALUES
(1006, 'Smartphone', 'Apple', 'iPhone 14', 'iOS', 7122, 'Active', NOW()),
(1007, 'Tablet', 'Samsung', 'Tab S9', 'Android', 5231, 'Active', NOW()),
(1008, 'Smartphone', 'OnePlus', 'OnePlus 12', 'Android', 8890, 'Active', NOW());

-- Updates: Changes in device ownership or OS upgrades
UPDATE telco_schema.dim_device
SET os = 'iOS 17.4', updated_at = NOW()
WHERE device_id = 1001;

UPDATE telco_schema.dim_device
SET owner_customer_id = 9999, updated_at = NOW()
WHERE device_id = 1002;

-- Soft Deletes: Marking device as Inactive (e.g., decommissioned)
UPDATE telco_schema.dim_device
SET status = 'Inactive', updated_at = NOW()
WHERE device_id = 1003;

In [0]:
df1 = spark.read.table("telecom_7405608425992653.telco_schema.dim_device").where("updated_at >coalesce((select max(updated_at) from telecom_7405608425992653.telco_bronze.device_bronze_table),'1970-01-01')")
# df1.show()
df1.write.format("parquet").save(f"{source_path}/device_source/device2")

In [0]:
%sql
-- =========================================
-- 4. THIRD RUN (MORE CDC CHANGES)
-- =========================================

-- Inserts
INSERT INTO telco_schema.dim_device (device_id, device_type, brand, model, os, owner_customer_id, status, updated_at) VALUES
(1009, 'Smartphone', 'Apple', 'iPhone 15 Pro', 'iOS', 4452, 'Active', NOW()),
(1010, 'Smartphone', 'Samsung', 'Galaxy Z Fold 5', 'Android', 3321, 'Active', NOW());

-- Updates: Re-activating a device or changing model details
UPDATE telco_schema.dim_device
SET status = 'Active', updated_at = NOW()
WHERE device_id = 1004;

UPDATE telco_schema.dim_device
SET model = 'iPhone 14 Pro Max', updated_at = NOW()
WHERE device_id = 1006;

-- Soft Deletes
UPDATE telco_schema.dim_device
SET status = 'Inactive', updated_at = NOW()
WHERE device_id = 1007;

In [0]:
df1 = spark.read.table("telecom_7405608425992653.telco_schema.dim_device").where("updated_at >coalesce((select max(updated_at) from telecom_7405608425992653.telco_bronze.device_bronze_table),'1970-01-01')")
# df1.show()
df1.write.format("parquet").save(f"{source_path}/device_source/device3")